## 어텐션
1. W_q, W_k, W_v, W_o 는 모델에 추가되어있는 어텐션 계층의 파라미터이다.
2. 입력 x에 W_q, W_k, W_v를 각각 행렬곱해서 Q, K, V를 만든다.
3. 이때, Q와 KT를 내적하여 토큰행끼리 곱해지도록 하고, 각 값을 √d_k로 나누어 조절 후 소프트맥스로 각 행의 점수를 정규화 한다(행의 데이터 합이 1). 이 결과를 A 행렬이라 한다.
4. A의 (i, j)의 값은 i 번째 토큰이 j번째 토큰에 영향을 받는 정도에 대한 값이다.
5. V는 출력토큰에 더해줄 입력 토큰의 성분을 나타내는 행렬이다.
6. A와 V를 행렬곱해서 V의 행들을 A의 비중대로 합친다. 이후 W_o를 행렬곱해 출력을 만든다.
7. W_q, W_k: 두 토큰의 벡터로 i가 j에 영향받는 정도를 계산하는 방법을 학습한다.
8. W_v, W_o: 입력의 어떤 성분을 출력으로 넘겨줄 지 학습한다.
9. A·V 하나는 헤드 하나의 출력이다. 헤드마다 다른 비율 분포로 다른 성분을 가져오기 위해 멀티헤드를 쓴다.
10. 헤드가 h개인 경우, A·V의 출력을 열방향으로 concat한 후, W_o를 곱해서 출력을 만든다. shape(A·V) = N x d_k, shape(W_o) = (h*d_k) x d, shape(output) = N x d

## 셀프어텐션
Q, K, V를 만들때, 같은 x를 사용한다.  
x의 각 토큰이 다른 토큰들을 참고해서 다시 계산할때 사용한다. (트랜스포머의 모든 계층에 들어감)  
RNN과 비교하면, RNN은 앞 토큰의 정보를 한단계씩 넘기므로, 멀리있는 정보를 유지하기 어렵다.  
셀프어텐션은 거리와 관계없이 모든 토큰 쌍의 점수를 계산하므로, 거리와 관계없이 멀리있는 토큰의 정보를 바로 가져올 수 있다.

## 크로스어텐션
Q는 x로 만들고, K, V는 다른 시퀀스인 y로 만든다.  
일반적으로 x는 출력이 만들어지는 쪽이고, 출력의 행은 x의 토큰 수와 같다.  
y는 출력에 더해질 내용을 주는 쪽이다.

예시
- 번역모델에서의 x: 현재까지 번역된 토큰 벡터, y: 인코딩된 원문 전체의 벡터
- 스테이블 디퓨전에서의 x: UNet 중간 단계의 특징 맵, y: CLIP 텍스트 인코더의 출력 벡터 (프롬프트)

## UNet with cross attention
UNet의 레이어마다 크로스어텐션을 붙이는 이유
1. 이미지의 영역마다 참고하는 단어 토큰을 다르게 하기 위함 (크로스어텐션)
2. 해상도 단계마다 단어를 다시 반영하기 위함 (레이어마다)

## 코드
### 03에서 가져온 부분
- `pos_encoding`, `ConvBlock`, `Diffuser`, VAE는 03과 같다.
- VAE는 합성곱만 쓰므로, 입력이 32×64로 바뀌어도 코드는 그대로이고 z만 8×16×4가 된다.
- 바뀌는 것은 `UNetCond` 하나다. 레이블 임베딩을 시점 임베딩에 더하던 방식을 없애고, 아래의 `UNetAttn`이 크로스어텐션으로 조건을 받는다.

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, TensorDataset
from torchvision import transforms
from tqdm import tqdm

NUM_CLASSES = 10
NULL_LABEL = NUM_CLASSES  # 레이블 0~9 다음 번호인 10을 '조건 없음' ∅로 쓴다.

def pos_encoding(ts, output_dim, device='cpu'):
    i = torch.arange(output_dim, device=device)
    div_term = 10000 ** (i / output_dim)
    angles = ts.to(device).float()[:, None] / div_term[None, :]
    v = torch.zeros(len(ts), output_dim, device=device)
    v[:, 0::2] = torch.sin(angles[:, 0::2])
    v[:, 1::2] = torch.cos(angles[:, 1::2])
    return v


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_embed_dim):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU()
        )
        self.mlp = nn.Sequential(
            nn.Linear(time_embed_dim, in_ch),
            nn.ReLU(),
            nn.Linear(in_ch, in_ch)
        )

    def forward(self, x, v):
        N, C, _, _ = x.shape
        v = self.mlp(v)
        v = v.view(N, C, 1, 1)
        y = self.convs(x + v)
        return y

class Diffuser:
    def __init__(self, num_timesteps=1000, beta_start=0.0001, beta_end=0.02, device='cpu'):
        self.num_timesteps = num_timesteps
        self.device = device
        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1 - self.betas
        self.alpha_bars = torch.cumprod(self.alphas, dim=0)

    def add_noise(self, x_0, t):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()
        alpha_bar = self.alpha_bars[t - 1].view(-1, 1, 1, 1)
        noise = torch.randn_like(x_0, device=self.device)
        x_t = torch.sqrt(alpha_bar) * x_0 + torch.sqrt(1 - alpha_bar) * noise
        return x_t, noise

    # CFG 노이즈 예측 ε̃ = ε̂(x_t, ∅) + w·(ε̂(x_t, y) − ε̂(x_t, ∅))를 계산하고, guidance_scale이 w다.
    def guided_noise(self, model, x, t, labels, guidance_scale):
        null_labels = torch.full_like(labels, NULL_LABEL)
        # eval 상태의 BatchNorm은 누적 통계를 쓰므로, 조건부 입력과 무조건 입력을 한 배치로 묶어 UNet을 한 번만 호출해도 결과가 같다.
        eps = model(torch.cat([x, x]), torch.cat([t, t]), torch.cat([labels, null_labels]))
        eps_cond, eps_uncond = eps.chunk(2)
        return eps_uncond + guidance_scale * (eps_cond - eps_uncond)

    def denoise(self, model, x, t, labels, guidance_scale):
        T = self.num_timesteps
        assert (t >= 1).all() and (t <= T).all()

        t_idx = t - 1
        alpha = self.alphas[t_idx].view(-1, 1, 1, 1)
        alpha_bar = self.alpha_bars[t_idx].view(-1, 1, 1, 1)
        alpha_bar_prev = self.alpha_bars[t_idx - 1].view(-1, 1, 1, 1)

        model.eval()
        with torch.no_grad():
            eps = self.guided_noise(model, x, t, labels, guidance_scale)
        model.train()

        noise = torch.randn_like(x, device=self.device)
        noise[t == 1] = 0

        mu = (x - ((1 - alpha) / torch.sqrt(1 - alpha_bar)) * eps) / torch.sqrt(alpha)
        std = torch.sqrt((1 - alpha) * (1 - alpha_bar_prev) / (1 - alpha_bar))
        return mu + noise * std

    def sample(self, model, labels, guidance_scale, x_shape):
        batch_size = len(labels)
        x = torch.randn((batch_size, *x_shape), device=self.device)

        for i in tqdm(range(self.num_timesteps, 0, -1)):
            t = torch.full((batch_size,), i, device=self.device, dtype=torch.long)
            x = self.denoise(model, x, t, labels, guidance_scale)
        # z는 0~1 범위의 값이 아니므로 clamp하지 않고, 디코더를 거친 이미지만 0~1로 다룬다.
        return x

class Encoder(nn.Module):
    def __init__(self, latent_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1),   # (N, 32, 16, 32)
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),  # (N, 64, 8, 16)
            nn.ReLU()
        )
        self.conv_mu = nn.Conv2d(64, latent_ch, 3, padding=1)      # (N, 4, 8, 16)
        self.conv_logvar = nn.Conv2d(64, latent_ch, 3, padding=1)  # (N, 4, 8, 16)

    def forward(self, x):
        h = self.convs(x)
        mu = self.conv_mu(h)
        sigma = torch.exp(0.5 * self.conv_logvar(h))
        return mu, sigma

class Decoder(nn.Module):
    def __init__(self, latent_ch):
        super().__init__()
        self.convs = nn.Sequential(
            nn.Conv2d(latent_ch, 64, 3, padding=1),              # (N, 64, 8, 16)
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),  # (N, 32, 16, 32)
            nn.ReLU(),
            nn.ConvTranspose2d(32, 1, 4, stride=2, padding=1),   # (N, 1, 32, 64)
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.convs(z)

def reparameterize(mu, sigma):
    eps = torch.randn_like(sigma)
    return mu + eps * sigma


class VAE(nn.Module):
    def __init__(self, latent_ch, kl_weight):
        super().__init__()
        self.encoder = Encoder(latent_ch)
        self.decoder = Decoder(latent_ch)
        self.kl_weight = kl_weight

    def get_loss(self, x):
        mu, sigma = self.encoder(x)
        z = reparameterize(mu, sigma)
        x_hat = self.decoder(z)

        L1 = F.mse_loss(x_hat, x, reduction='sum')
        L2 = -torch.sum(1 + torch.log(sigma ** 2) - mu ** 2 - sigma ** 2)
        return (L1 + self.kl_weight * L2) / len(x)

### 데이터: 숫자 두 개를 나란히 붙인 이미지
- 조건이 레이블 하나면 모든 위치에서 비율이 1이 되므로, 크로스어텐션이 레이블 임베딩을 더하는 것과 같은 계산이 된다.
- 그래서 MNIST 숫자 두 개를 나란히 붙인 32×64 이미지를 쓰고, "3 7"처럼 토큰 두 개를 조건으로 준다.
- 왼쪽 위치는 첫 번째 토큰을, 오른쪽 위치는 두 번째 토큰을 참고해야 레이블대로 그릴 수 있다.
- VAE 학습은 03과 같다.

In [ ]:
batch_size = 128
latent_ch = 4
vae_epochs = 5
vae_lr = 1e-3
kl_weight = 0.01
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else torch.device('cpu')


class TwoDigitMNIST(Dataset):
    def __init__(self, base, seed=0):
        self.base = base
        generator = torch.Generator().manual_seed(seed)
        self.partner = torch.randperm(len(base), generator=generator)  # i번째 숫자의 오른쪽에 붙일 숫자의 번호

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        left, left_label = self.base[i]
        right, right_label = self.base[self.partner[i].item()]
        image = torch.cat([left, right], dim=2)           # (1, 32, 64)
        labels = torch.tensor([left_label, right_label])  # (2,)
        return image, labels


preprocess = transforms.Compose([transforms.Pad(2), transforms.ToTensor()])  # (1, 32, 32)
mnist = torchvision.datasets.MNIST(root='./data', download=True, transform=preprocess)
dataset = TwoDigitMNIST(mnist)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

vae = VAE(latent_ch, kl_weight).to(device)
vae_optimizer = Adam(vae.parameters(), lr=vae_lr)

for epoch in range(vae_epochs):
    loss_sum = 0.0
    for images, _ in tqdm(dataloader):
        x = images.to(device)
        vae_optimizer.zero_grad()
        loss = vae.get_loss(x)
        loss.backward()
        vae_optimizer.step()
        loss_sum += loss.item()
    print(f'Epoch {epoch} | Loss: {loss_sum / len(dataloader)}')

### 어텐션을 붙인 UNet
- `Attention`은 위의 정리 1~10번을 그대로 코드로 옮긴 것이다. `context`를 주지 않으면 셀프어텐션이고, 주면 크로스어텐션이다.
- `TransformerBlock`은 SD의 블록과 같은 순서로 셀프어텐션, 크로스어텐션, MLP를 계산하고, 각 결과를 입력에 더한다.
  - 특징 맵 (N, C, H, W)를 (N, H·W, C)로 바꿔서, 위치 하나를 토큰 하나로 다룬다.
- `UNetAttn`은 02의 `UNetCond`에서 ConvBlock마다 `TransformerBlock`을 붙인 구조다. 8×16, 4×8, 2×4의 세 해상도에서 조건을 다시 참고한다.
- 조건 y는 토큰 임베딩에 위치 임베딩을 더해 만든다. 크로스어텐션의 출력은 y의 순서를 바꿔도 같으므로, 위치 임베딩이 없으면 "3 7"과 "7 3"을 구분하지 못한다.
  - SD에서는 이 자리에 CLIP 텍스트 인코더의 출력이 들어간다.

In [ ]:
class Attention(nn.Module):
    def __init__(self, query_dim, context_dim, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.to_q = nn.Linear(query_dim, query_dim, bias=False)    # W_q
        self.to_k = nn.Linear(context_dim, query_dim, bias=False)  # W_k
        self.to_v = nn.Linear(context_dim, query_dim, bias=False)  # W_v
        self.to_out = nn.Linear(query_dim, query_dim)              # W_o
        self.attn_map = None

    def forward(self, x, context=None):
        context = x if context is None else context  # context가 없으면 셀프어텐션, 있으면 크로스어텐션이다.
        N, L, d = x.shape
        h = self.num_heads
        d_k = d // h

        q = self.to_q(x).view(N, L, h, d_k).transpose(1, 2)        # Q: (N, h, L, d_k)
        k = self.to_k(context).view(N, -1, h, d_k).transpose(1, 2)  # K: (N, h, M, d_k)
        v = self.to_v(context).view(N, -1, h, d_k).transpose(1, 2)  # V: (N, h, M, d_k)

        scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)  # S = Q·Kᵀ/√d_k: (N, h, L, M)
        attn = scores.softmax(dim=-1)                       # A: 행마다 합이 1
        self.attn_map = attn.detach()
        out = attn @ v                                      # A·V: (N, h, L, d_k)
        out = out.transpose(1, 2).reshape(N, L, d)          # 헤드 h개를 열 방향으로 concat: (N, L, h·d_k)
        return self.to_out(out)


class TransformerBlock(nn.Module):
    def __init__(self, ch, context_dim, num_heads=4):
        super().__init__()
        self.norm_in = nn.GroupNorm(8, ch)
        self.norm1 = nn.LayerNorm(ch)
        self.self_attn = Attention(ch, ch, num_heads)
        self.norm2 = nn.LayerNorm(ch)
        self.cross_attn = Attention(ch, context_dim, num_heads)
        self.norm3 = nn.LayerNorm(ch)
        self.ff = nn.Sequential(nn.Linear(ch, 4 * ch), nn.GELU(), nn.Linear(4 * ch, ch))

    def forward(self, x, context):
        N, C, H, W = x.shape
        h = self.norm_in(x).flatten(2).transpose(1, 2)   # (N, H·W, C): 위치 하나가 토큰 하나
        h = h + self.self_attn(self.norm1(h))             # 위치끼리 참고한다.
        h = h + self.cross_attn(self.norm2(h), context)   # 위치마다 조건 토큰을 참고한다.
        h = h + self.ff(self.norm3(h))
        h = h.transpose(1, 2).reshape(N, C, H, W)
        return x + h


class UNetAttn(nn.Module):
    def __init__(self, in_ch=4, time_embed_dim=100, context_dim=64, num_tokens=2):
        super().__init__()
        self.time_embed_dim = time_embed_dim
        self.token_emb = nn.Embedding(NUM_CLASSES + 1, context_dim)        # 레이블 0~9와 ∅
        self.pos_emb = nn.Parameter(torch.zeros(num_tokens, context_dim))  # 토큰 순서

        self.down1 = ConvBlock(in_ch, 64, time_embed_dim)
        self.attn_down1 = TransformerBlock(64, context_dim)
        self.down2 = ConvBlock(64, 128, time_embed_dim)
        self.attn_down2 = TransformerBlock(128, context_dim)
        self.bot1 = ConvBlock(128, 256, time_embed_dim)
        self.attn_bot1 = TransformerBlock(256, context_dim)
        self.up2 = ConvBlock(256 + 128, 128, time_embed_dim)
        self.attn_up2 = TransformerBlock(128, context_dim)
        self.up1 = ConvBlock(128 + 64, 64, time_embed_dim)
        self.attn_up1 = TransformerBlock(64, context_dim)
        self.out = nn.Conv2d(64, in_ch, 1)

        self.maxpool = nn.MaxPool2d(2)
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear')

    def forward(self, x, timesteps, labels):
        t = pos_encoding(timesteps, self.time_embed_dim, device=x.device)
        context = self.token_emb(labels) + self.pos_emb  # y: (N, 2, context_dim)

        x1 = self.attn_down1(self.down1(x, t), context)  # (N, 64, 8, 16)
        x = self.maxpool(x1)
        x2 = self.attn_down2(self.down2(x, t), context)  # (N, 128, 4, 8)
        x = self.maxpool(x2)

        x = self.attn_bot1(self.bot1(x, t), context)     # (N, 256, 2, 4)

        x = self.upsample(x)
        x = torch.cat([x, x2], dim=1)
        x = self.attn_up2(self.up2(x, t), context)       # (N, 128, 4, 8)
        x = self.upsample(x)
        x = torch.cat([x, x1], dim=1)
        x = self.attn_up1(self.up1(x, t), context)       # (N, 64, 8, 16)

        x = self.out(x)
        return x

### z로 UNet 학습
- 03과 같이 VAE를 고정하고 데이터셋 전체를 한 번 인코딩한다.
- CFG를 위해 배치의 10%는 조건 토큰 두 개를 함께 ∅로 바꾼다. SD에서 빈 프롬프트를 쓰는 것과 같다.

In [ ]:
num_timesteps = 1000
epochs = 10
lr = 1e-3
p_uncond = 0.1

vae.requires_grad_(False)
with torch.no_grad():
    latents, latent_labels = [], []
    for images, labels in DataLoader(dataset, batch_size=1000):
        mu, _ = vae.encoder(images.to(device))
        latents.append(mu)
        latent_labels.append(labels)
latents = torch.cat(latents)                         # (60000, 4, 8, 16)
latent_labels = torch.cat(latent_labels).to(device)  # (60000, 2)

scale_factor = 1 / latents.std()
print(f'z std: {latents.std():.3f}, scale_factor: {scale_factor:.3f}')
latent_loader = DataLoader(TensorDataset(latents * scale_factor, latent_labels), batch_size=batch_size, shuffle=True)

diffuser = Diffuser(num_timesteps, device=device)
model = UNetAttn(in_ch=latent_ch).to(device)
optimizer = Adam(model.parameters(), lr=lr)

losses = []
for epoch in range(epochs):
    loss_sum = 0.0
    cnt = 0

    for z, labels in tqdm(latent_loader):
        optimizer.zero_grad()
        is_dropped = torch.rand(len(labels), device=device) < p_uncond
        labels = torch.where(is_dropped[:, None], NULL_LABEL, labels)  # 조건 토큰 두 개를 함께 ∅로 바꾼다.
        t = torch.randint(1, num_timesteps + 1, (len(z),), device=device)

        z_noisy, noise = diffuser.add_noise(z, t)
        noise_pred = model(z_noisy, t, labels)
        loss = F.mse_loss(noise, noise_pred)

        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        cnt += 1

    losses.append(loss_sum / cnt)
    print(f'Epoch {epoch} | Loss: {losses[-1]}')

plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.show()

### 생성
- 같은 숫자 두 개를 순서만 바꿔 넣는다. "3 7"과 "7 3"이 다르게 나오면, 모델이 토큰 순서와 위치를 함께 쓰고 있다는 뜻이다.

In [ ]:
def show_pairs(rows, row_names, pairs):
    fig, axes = plt.subplots(len(rows), len(pairs), figsize=(len(pairs) * 1.6, len(rows) * 1.0), squeeze=False)
    for r, (images, name) in enumerate(zip(rows, row_names)):
        for c, image in enumerate(images):
            ax = axes[r, c]
            ax.imshow(image[0], cmap='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            if r == 0:
                ax.set_title(f'{pairs[c][0]} {pairs[c][1]}', fontsize=9)
        axes[r, 0].set_ylabel(name, fontsize=8)
    plt.tight_layout()
    plt.show()


guidance_scale = 3.0
pairs = [[3, 7], [7, 3], [0, 1], [1, 0], [2, 5], [5, 2], [4, 9], [9, 4], [6, 8], [8, 6]]
labels = torch.tensor(pairs, device=device)

torch.manual_seed(0)
z = diffuser.sample(model, labels, guidance_scale, x_shape=(latent_ch, 8, 16))  # (10, 4, 8, 16)
with torch.no_grad():
    images = vae.decoder(z / scale_factor).cpu()                                 # (10, 1, 32, 64)

show_pairs([images[0::2], images[1::2]], ['a b', 'b a'], pairs[0::2])

### 크로스어텐션의 A 보기
- 생성한 z에 t = 300만큼 노이즈를 섞어 UNet을 한 번 호출하고, 다섯 단계의 크로스어텐션에서 A를 모두 꺼낸다.
- 토큰이 두 개이므로 A의 각 행은 [a, 1 − a]이다. 그래서 첫 번째 토큰의 열만 해상도에 맞춰 배치해서 본다.
  - 밝은 위치일수록 첫 번째 토큰을 많이 참고했고, 어두운 위치일수록 두 번째 토큰을 많이 참고했다.
  - 헤드 4개의 A를 평균해서 본다.
- 왼쪽 절반과 오른쪽 절반의 밝기가 다른 단계가, 어느 숫자를 어느 쪽에 그릴지 정하는 단계다.

In [ ]:
levels = ['attn_down1', 'attn_down2', 'attn_bot1', 'attn_up2', 'attn_up1']
t = torch.full((len(z),), 300, device=device)
z_t, _ = diffuser.add_noise(z, t)
model.eval()
with torch.no_grad():
    model(z_t, t, labels)

fig, axes = plt.subplots(4, 1 + len(levels), figsize=(12, 4.5))
for r in range(4):
    axes[r, 0].imshow(images[r, 0], cmap='gray')
    axes[r, 0].set_ylabel(f'{pairs[r][0]} {pairs[r][1]}')
    for c, level in enumerate(levels, start=1):
        attn = getattr(model, level).cross_attn.attn_map.mean(dim=1)  # 헤드 평균 A: (10, 위치 수, 2)
        H = int(math.sqrt(attn.shape[1] // 2))                       # 위치 수는 H × 2H다.
        first_map = attn[r, :, 0].reshape(H, 2 * H).cpu()            # 첫 번째 토큰의 열
        axes[r, c].imshow(first_map, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
        if r == 0:
            axes[r, c].set_title(f'{level}\n{H}x{2 * H}', fontsize=8)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

### 05를 위해 저장
- 05에서 같은 모델에 샘플러만 바꿔 보기 위해 UNet과 디코더를 저장한다.
- `data/`는 .gitignore에 있으므로 커밋되지 않는다.

In [ ]:
torch.save({
    'unet': model.state_dict(),
    'decoder': vae.decoder.state_dict(),
    'scale_factor': scale_factor.item(),
}, './data/04_unet_attention.pt')